# SIRA and CoDA Comparison on Colab L4

This notebook compares SIRA and CoDA using the same balanced six-model matrix:

- Llama: 3B and 8B
- Gemma 4: E2B and E4B
- Qwen 2.5: 3B and 7B

The six models are attack/rewrite models. SIRA and CoDA attack the same 500 shared KGW-watermarked texts generated with OPT-1.3B.

Before running, choose **Runtime > Change runtime type > L4 GPU**. Add a Colab Secret named `HF_TOKEN` with accepted access to any gated Llama or Gemma checkpoints. Inaccessible models are recorded as skipped rather than crashing the run.


In [ ]:
# Check that Colab assigned the requested L4 GPU.
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", gpu_name)
if "L4" not in gpu_name:
    raise RuntimeError("This notebook requires an L4 GPU. Change the Colab runtime type and try again.")


In [ ]:
# Experiment settings for the full 500-sample comparison.
REPO_URL = "https://github.com/hanifnoerr/Self-information-Rewrite-Attack.git"
BRANCH = "codex/browser-colab-l4"
REPO_DIR = "/content/Self-information-Rewrite-Attack"
OUTPUT_ROOT = "/content/drive/MyDrive/sira_500_outputs"

ALGORITHM = "KGW"
SAMPLES = 500
RESET_OUTPUTS = False
MATRIX_CONFIG = f"{REPO_DIR}/config/model_matrix_l4.json"

print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {SAMPLES}")
print("Models: Llama, Gemma 4, and Qwen; two sizes each")


In [ ]:
# Clone the adapted repository into /content.
from pathlib import Path
import shutil
import subprocess

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
print("Cloned repository to:", REPO_DIR)


In [ ]:
# Install requirements.
subprocess.run(
    ["pip", "install", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
subprocess.run(
    ["pip", "install", "--upgrade", "transformers", "accelerate"],
    check=True,
)
print("Requirements installed.")


In [ ]:
# Log in once. Gated Llama or Gemma checkpoints require accepted Hugging Face access.
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Using the HF_TOKEN Colab Secret.")
else:
    print("No HF_TOKEN found. Gated model runs will be marked skipped.")


In [ ]:
# Mount Drive first so the long run can resume after a Colab disconnect.
import os
from google.colab import drive

drive.mount("/content/drive")

if RESET_OUTPUTS and Path(OUTPUT_ROOT).exists():
    shutil.rmtree(OUTPUT_ROOT)
    print("Removed old outputs so every model uses the same fresh data.")

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

models_config_path = f"{OUTPUT_ROOT}/model_runs.json"
coda_models_config_path = f"{OUTPUT_ROOT}/coda_model_runs.json"
env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["ALGORITHM"] = ALGORITHM
env["SAMPLES"] = str(SAMPLES)
env["OUTPUT_ROOT"] = OUTPUT_ROOT


In [ ]:
# Run SIRA with all six attack models.
subprocess.run(
    [
        "python", "scripts/run_model_matrix_l4.py",
        "--config_path", MATRIX_CONFIG,
        "--repo_dir", REPO_DIR,
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

# Run CoDA with the same six attack models.
subprocess.run(
    [
        "python", "scripts/run_coda_matrix_l4.py",
        "--config_path", MATRIX_CONFIG,
        "--repo_dir", REPO_DIR,
        "--input_path", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--output_root", OUTPUT_ROOT,
        "--threshold", "30",
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)
print("CoDA model status:", coda_models_config_path)

# Update the environment record with both SIRA and CoDA model statuses.
subprocess.run(
    ["python", "scripts/write_environment.py", "--output_path", f"{OUTPUT_ROOT}/environment.json", "--models_config", models_config_path, "--coda_models_config", coda_models_config_path],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Evaluate attack success and semantic preservation against the paper values.
subprocess.run(
    [
        "python", "scripts/evaluate_sira_transfer.py",
        "--generation_model", "facebook/opt-1.3b",
        "--algorithm", ALGORITHM,
        "--watermarked_input", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--models_config", models_config_path,
        "--coda_models_config", coda_models_config_path,
        "--output_root", OUTPUT_ROOT,
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

subprocess.run(
    [
        "python", "scripts/compare_transfer_results.py",
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Display the final report and paper-style comparison dataframe.
import pandas as pd
from IPython.display import display

report_path = Path(OUTPUT_ROOT) / "final_report.md"
print(report_path.read_text(encoding="utf-8"))

comparison_dataframe = pd.read_csv(f"{OUTPUT_ROOT}/results/paper_style_comparison.csv")
comparison_columns = [
    "attack_method", "method", "paper_method", "model_family", "size_tier", "parameter_size", "run_status",
    "paper_attack_success_rate", "reproduced_attack_success_rate", "difference", "coda_minus_sira_asr",
    "semantic_similarity", "average_anchor_count", "average_anchor_rate", "note",
]
display(comparison_dataframe[comparison_columns])

print("\nSaved output files:")
for path in sorted(Path(OUTPUT_ROOT).rglob("*")):
    if path.is_file():
        print(path)


In [ ]:
# Outputs are already stored in Drive throughout the run.
print("Persistent output directory:", OUTPUT_ROOT)
print("Re-run the notebook with RESET_OUTPUTS = False to resume partial outputs.")


## Stop the Runtime

After the final report is saved in Drive, use **Runtime > Disconnect and delete runtime** to release the L4 GPU.
